<a href="https://colab.research.google.com/github/hss2301/IS-507/blob/Collaboration/Data_onboard_and_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import pandas as pd
from google.colab import files

# 1. Prompt you to upload your CSV files (you can select multiple files at once)
print("Please select your CSV files to upload:")
uploaded = files.upload()

# 2. Dictionary to store the dataframes
dfs = {}

# 3. Loop through all uploaded files and load them into pandas
for filename in uploaded.keys():
    if filename.endswith('.csv'):
        # Remove the '.csv' extension to use as the dataframe key name
        df_name = os.path.splitext(filename)[0]

        # Read the uploaded bytes directly into pandas
        dfs[df_name] = pd.read_csv(filename)
        print(f"Loaded: {filename} -> DataFrame named: dfs['{df_name}']")

Please select your CSV files to upload:


Saving cattlecensus.csv to cattlecensus.csv
Saving district level cropwise irrigation.csv to district level cropwise irrigation.csv
Saving evapotranspirationpotential.csv to evapotranspirationpotential.csv
Saving gdp.csv to gdp.csv
Saving ICRISAT-District Level Data  filed labour wages.csv to ICRISAT-District Level Data  filed labour wages.csv
Saving ICRISAT-District Level Data -operational holdings.csv to ICRISAT-District Level Data -operational holdings.csv
Saving ICRISAT-District Level Data Evapotranspiration.csv to ICRISAT-District Level Data Evapotranspiration.csv
Saving ICRISAT-District Level Data input fertilizer consumption.csv to ICRISAT-District Level Data input fertilizer consumption.csv
Saving ICRISAT-District Level Data Population.csv to ICRISAT-District Level Data Population.csv
Saving ICRISAT-District Level Data Price.csv to ICRISAT-District Level Data Price.csv
Saving ICRISAT-District Level Data Soil type.csv to ICRISAT-District Level Data Soil type.csv
Saving ICRISAT-D

In [5]:
import pandas as pd
import numpy as np

def detailed_eda(df_dict):
    for name, df in df_dict.items():
        print("=" * 80)
        print(f"DATASET: {name}")
        print("=" * 80)
        print(f"Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}")
        print("-" * 80)

        summary_data = []

        for col in df.columns:
            dtype = df[col].dtype
            unique_count = df[col].nunique()

            # Count standard NaNs
            nan_count = df[col].isna().sum()

            # Count -1 values as missing if the column is numeric or can be evaluated
            minus_one_count = 0
            try:
                # Check for -1 in numeric types
                minus_one_count = (df[col] == -1).sum()
            except Exception:
                # Fallback for mixed types or strings if necessary
                pass

            total_missing = nan_count + minus_one_count
            missing_pct = (total_missing / len(df)) * 100 if len(df) > 0 else 0

            summary_data.append({
                'Column': col,
                'Type': dtype,
                'Unique Values': unique_count,
                'Missing (NaN + -1)': total_missing,
                'Missing %': f"{missing_pct:.2f}%"
            })

        # Convert summary list to a clean DataFrame for display
        summary_df = pd.DataFrame(summary_data)

        # Display options to show all columns nicely in Colab
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)

        print(summary_df)
        print("\n" + "\n")

# Run the EDA function on your dictionary of dataframes
detailed_eda(dfs)

DATASET: cattlecensus
Total Rows: 2915 | Total Columns: 13
--------------------------------------------------------------------------------
                                   Column     Type  Unique Values  Missing (NaN + -1) Missing %
0                               Dist Code    int64            591                   0     0.00%
1                                    Year    int64             11                   0     0.00%
2                              State Code    int64             20                   0     0.00%
3                              State Name   object             20                   0     0.00%
4                               Dist Name   object            586                   0     0.00%
5      ADULT MALE CROSSBRED (1000 Number)  float64            773                 546    18.73%
6     ADULT MALE INDIGENOUS (1000 Number)  float64           2109                 546    18.73%
7    ADULT FEMALE CROSSBRED (1000 Number)  float64           1750                 546    18.

In [7]:
import pandas as pd
import numpy as np

def prepare_merged_datasets(df_dict):
    merge_keys = ['Dist Code', 'Year', 'State Code', 'State Name', 'Dist Name']

    # Helper function to standardize keys, clean -1 values, and convert to NaN
    def clean_df(df):
        df_clean = df.copy()

        # Standardize merge key types to prevent join errors
        if 'Year' in df_clean.columns:
            # Convert Year to string first to catch mixed string/int types uniformly, or int if possible.
            # Let's standardize Year and Dist Code as string/int safely:
            df_clean['Year'] = df_clean['Year'].astype(str).str.strip()
        if 'Dist Code' in df_clean.columns:
            df_clean['Dist Code'] = df_clean['Dist Code'].astype(str).str.strip()

        # Replace -1 with NaN in numeric columns
        for col in df_clean.select_dtypes(include=[np.number]).columns:
            if col not in merge_keys:
                df_clean[col] = df_clean[col].replace(-1, np.nan)
        return df_clean

    print("--- Preprocessing and Cleaning All Dataframes ---")
    cleaned_dict = {name: clean_df(df) for name, df in df_dict.items()}

    print("--- Building Dataset 1: Agronomic Output & Agro-Meteorology ---")
    crop_df = cleaned_dict['ICRISAT-District Level Dat-crop']

    met_datasets = [
        ('tempmax', cleaned_dict['tempmax']),
        ('tempmin', cleaned_dict['tempmin']),
        ('precipitation', cleaned_dict['precipitation']),
        ('evapotranspirationpotential', cleaned_dict['evapotranspirationpotential']),
        ('Waterdefecit', cleaned_dict['ICRISAT-District Level Data Waterdefecit'])
    ]

    agronomic_met_df = crop_df
    for name, df in met_datasets:
        cols_to_use = [c for c in df.columns if c not in ['State Name', 'Dist Name', 'State Code'] or c in merge_keys]
        agronomic_met_df = pd.merge(agronomic_met_df, df[cols_to_use], on=merge_keys, how='left')

    print(f"Dataset 1 Created Successfully! Shape: {agronomic_met_df.shape}")

    print("\n--- Building Dataset 2: Agronomic Output & Socio-Economic Infrastructure ---")
    socio_datasets = [
        ('Population', cleaned_dict['ICRISAT-District Level Data Population']),
        ('gdp', cleaned_dict['gdp']),
        ('Bank Office', cleaned_dict['ICRISAT-District Level Data-Infrastructure-Bank Office']),
        ('Market Road', cleaned_dict['ICRISAT-District Level Data-Infrastructure-Market Road']),
        ('operational holdings', cleaned_dict['ICRISAT-District Level Data -operational holdings'])
    ]

    agronomic_socio_df = crop_df
    for name, df in socio_datasets:
        cols_to_use = [c for c in df.columns if c not in ['State Name', 'Dist Name', 'State Code'] or c in merge_keys]
        agronomic_socio_df = pd.merge(agronomic_socio_df, df[cols_to_use], on=merge_keys, how='left')

    print(f"Dataset 2 Created Successfully! Shape: {agronomic_socio_df.shape}")

    return agronomic_met_df, agronomic_socio_df

# Execute the updated pipeline
agronomic_met_final, agronomic_socio_final = prepare_merged_datasets(dfs)

--- Preprocessing and Cleaning All Dataframes ---
--- Building Dataset 1: Agronomic Output & Agro-Meteorology ---
Dataset 1 Created Successfully! Shape: (18060, 139)

--- Building Dataset 2: Agronomic Output & Socio-Economic Infrastructure ---
Dataset 2 Created Successfully! Shape: (18060, 170)


In [8]:
import pandas as pd
import numpy as np

def prepare_and_export_datasets(df_dict):
    merge_keys = ['Dist Code', 'Year', 'State Code', 'State Name', 'Dist Name']

    # Helper function to standardize keys, clean -1 values, and convert to NaN
    def clean_df(df):
        df_clean = df.copy()
        if 'Year' in df_clean.columns:
            df_clean['Year'] = df_clean['Year'].astype(str).str.strip()
        if 'Dist Code' in df_clean.columns:
            df_clean['Dist Code'] = df_clean['Dist Code'].astype(str).str.strip()

        for col in df_clean.select_dtypes(include=[np.number]).columns:
            if col not in merge_keys:
                df_clean[col] = df_clean[col].replace(-1, np.nan)
        return df_clean

    print("--- Preprocessing and Cleaning All Dataframes ---")
    cleaned_dict = {name: clean_df(df) for name, df in df_dict.items()}

    crop_df = cleaned_dict['ICRISAT-District Level Dat-crop']

    # 1. Extended Agronomic & Agro-Meteorology (with inputs, soil, and irrigation)
    print("--- Building Dataset 1: Agronomic-Met-ExtInput Data ---")
    met_ext_datasets = [
        ('tempmax', cleaned_dict['tempmax']),
        ('tempmin', cleaned_dict['tempmin']),
        ('precipitation', cleaned_dict['precipitation']),
        ('evapotranspirationpotential', cleaned_dict['evapotranspirationpotential']),
        ('Waterdefecit', cleaned_dict['ICRISAT-District Level Data Waterdefecit']),
        ('fertilizer consumption', cleaned_dict['ICRISAT-District Level Data input fertilizer consumption']),
        ('Soil type', cleaned_dict['ICRISAT-District Level Data Soil type']),
        ('cropwise irrigation', cleaned_dict['district level cropwise irrigation'])
    ]

    agronomic_met_extinput_df = crop_df
    for name, df in met_ext_datasets:
        cols_to_use = [c for c in df.columns if c not in ['State Name', 'Dist Name', 'State Code'] or c in merge_keys]
        agronomic_met_extinput_df = pd.merge(agronomic_met_extinput_df, df[cols_to_use], on=merge_keys, how='left')

    print(f"Agronomic-Met-ExtInput Created Successfully! Shape: {agronomic_met_extinput_df.shape}")

    # 2. Agronomic & Socio-Economic Infrastructure
    print("\n--- Building Dataset 2: Agronomic Output & Socio-Economic Infrastructure ---")
    socio_datasets = [
        ('Population', cleaned_dict['ICRISAT-District Level Data Population']),
        ('gdp', cleaned_dict['gdp']),
        ('Bank Office', cleaned_dict['ICRISAT-District Level Data-Infrastructure-Bank Office']),
        ('Market Road', cleaned_dict['ICRISAT-District Level Data-Infrastructure-Market Road']),
        ('operational holdings', cleaned_dict['ICRISAT-District Level Data -operational holdings'])
    ]

    agronomic_socio_df = crop_df
    for name, df in socio_datasets:
        cols_to_use = [c for c in df.columns if c not in ['State Name', 'Dist Name', 'State Code'] or c in merge_keys]
        agronomic_socio_df = pd.merge(agronomic_socio_df, df[cols_to_use], on=merge_keys, how='left')

    print(f"Dataset 2 Created Successfully! Shape: {agronomic_socio_df.shape}")

    # 3. Export to CSV
    print("\n--- Exporting Datasets to CSV ---")
    agronomic_met_extinput_df.to_csv('agronomic_met_extinput_final.csv', index=False)
    agronomic_socio_df.to_csv('agronomic_socio_final.csv', index=False)
    print("Files successfully exported: 'agronomic_met_extinput_final.csv' and 'agronomic_socio_final.csv'")

    return agronomic_met_extinput_df, agronomic_socio_df

# Execute pipeline and save files
agronomic_met_extinput_final, agronomic_socio_final = prepare_and_export_datasets(dfs)

--- Preprocessing and Cleaning All Dataframes ---
--- Building Dataset 1: Agronomic-Met-ExtInput Data ---
Agronomic-Met-ExtInput Created Successfully! Shape: (18060, 175)

--- Building Dataset 2: Agronomic Output & Socio-Economic Infrastructure ---
Dataset 2 Created Successfully! Shape: (18060, 170)

--- Exporting Datasets to CSV ---
Files successfully exported: 'agronomic_met_extinput_final.csv' and 'agronomic_socio_final.csv'
